# Predicting Content Virality on YouTube/TikTok
## Binary Classification: Predicting "Rising" Content Trends (2025)

**Target Variable:** `target = (trend_label == "rising").astype(int)`

---
This project implements a supervised binary classification framework to predict the virality of social media content. By leveraging metadata and performance metrics from YouTube and TikTok, the model aims to distinguish between "rising" content—classified as viral—and non-viral content. The analysis adheres to a rigorous machine learning lifecycle, prioritizing methodological integrity, data quality, and model interpretability.


## 1 Environment Setup

All library imports and environment configurations are consolidated in this section to ensure the notebook's reproducibility and streamline the dependency management process

### 1.1 Imports

In [ ]:
# -- Paths --- 
from pathlib import Path

# --- Data manipulation ---
import numpy as np
import pandas as pd

# --- Visualization ---
import matplotlib.pyplot as plt
import seaborn as sns

# --- Statistics test ---
from scipy.stats import chi2_contingency, mannwhitneyu

# --- Scikit-Learn: partitioning e validation ---
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold

# --- Scikit-Learn: preprocessing e pipeline ---
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# --- Scikit-Learn: models ---
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier

# --- Scikit-Learn: metrics ---
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay, accuracy_score, precision_score,
    recall_score, f1_score, roc_auc_score, roc_curve, classification_report,
)

# --- Explainable AI ---
import shap

### 1.2 Configuration

In [ ]:
# --- Replicability ---
RANDOM_STATE = 107

# --- Visualization settings ---
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (9, 5)
np.random.seed(RANDOM_STATE)

print("Setup completed.")
print(f"  pandas : {pd.__version__}")
print(f"  numpy  : {np.__version__}")
print(f"  shap   : {shap.__version__}")

## 2. Data ingestion & Partitionig

The partitioning is carried out immediately after loading, prior to any in-depth inspection, imputation, or scaling.The reason is methodological. 

The split is stratified on the target to preserve class proportions. 

One column is removed prior to any other consideration: row_id — a unique identifier. It conveys no generalizable information and, in a tree-based model, only provides opportunities to memorize the Training Set.

### 2.1 Data import

In [ ]:
# --- Dataset import ---
dataset_path = Path("Data") / "raw_tiktok_youtube_trends.csv"
datadict_path = Path("Data") / "DATA_DICTIONARY.csv"

df_raw = pd.read_csv(dataset_path, encoding="utf-8-sig")
data_dictionary = pd.read_csv(datadict_path)

print(f"Raw dataset dimensions: {df_raw.shape[0]} righe x {df_raw.shape[1]} colonne")
df_raw.head(10)

### 2.2 Split Train/Test

In [ ]:
# --- Target definition
target = (df_raw["trend_label"] == "rising").astype(int)

# --- Deletion of 'row-id' column---
df = df_raw.drop(columns=["row_id"])

# --- Deduplication ---
n_duplicates = df.duplicated().sum()
df = df.drop_duplicates().reset_index(drop=True)
print(f"Duplicated rows removed: {n_duplicates}")

# --- Split of predictors / target ---
X = df.drop(columns=["trend_label"])
y = target

# --- Stratified split 80/20 ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.20, 
    stratify=y, 
    random_state=RANDOM_STATE,
)

print(f"\nTraining Set : {X_train.shape[0]} rows")
print(f"Test Set     : {X_test.shape[0]} rows")
print(f"Target/Train proportion: {y_train.mean()*100:.3f} %  || test: {y_test.mean()*100:.3f} %")

## 3 Exploratory Data Analysis & Data Quality

All subsequent EDA is conducted exclusively on the Training Set. 